## 03 — Gold Layer: Aggregation & Analysis-Ready Tables
**Owners: Yanan Y**

Reads all Silver tables, joins and aggregates them into Gold Delta tables
ready for direct dashboard queries and EDA.

**Contents:**

- Taxonomy join (enriched line items) 
- gold_spend_by_vendor 
- gold_spend_by_department 
- gold_monthly_trend 
- gold_payment_cycle 
- gold_spend_by_taxonomy 

**Dependancy:** Requires `02_silver_processing.py` to be complete first.

### Imports

In [0]:
import logging

from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

# ── Logging setup ──────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("gold")

CATALOG = "healthcare_procurement"
SCHEMA  = "procurement_analytics"

log.info("Imports complete. Catalog=%s  Schema=%s", CATALOG, SCHEMA)

16:30:30  INFO  Imports complete. Catalog=healthcare_procurement  Schema=procurement_analytics


### Setup

In [0]:
# All schema and catalog are already created in 01 and 02, just reuse them
# Verify existing structures instead of creating new ones
try:
    spark.sql(f"USE CATALOG {CATALOG}")
    spark.sql(f"USE SCHEMA {SCHEMA}")
    log.info("Successfully connected to Catalog: %s and Schema: %s", CATALOG, SCHEMA)
except Exception as error:
    raise RuntimeError("The catalog or schema does not exist. Please run step 01 and step 02 first.")

log.info("Setup complete. Environment verified.")

16:30:55  INFO  Successfully connected to Catalog: healthcare_procurement and Schema: procurement_analytics
16:30:55  INFO  Setup complete. Environment verified.


### Helper Functions

In [0]:
# Reuseable function to write and read tables

def write_table(df, table_name: str, mode: str = "overwrite") -> None:
    """Write a Spark DataFrame as a Delta table in the active schema."""
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    df.write.format("delta").mode(mode).saveAsTable(full_name)
    log.info("Written -> %s  (%d rows)", full_name, df.count())


def read_table(table_name: str):
    """Read a Delta table from the active schema and return a Spark DataFrame."""
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    df = spark.read.table(full_name)
    log.info("Loaded  <- %s  (%d rows)", full_name, df.count())
    return df

log.info("Helper functions defined.")

16:30:56  INFO  Helper functions defined.


---
## Gold Layer — Joined & Aggregated Data
### Read Silver Tables

In [0]:
# Load Silver tables as the starting point for Gold aggregations
vendors_s  = read_table("silver_vendors")
po_s       = read_table("silver_purchase_orders")
lines_s    = read_table("silver_line_items")
taxonomy_s = read_table("silver_spend_taxonomy")

16:31:05  INFO  Loaded  <- healthcare_procurement.procurement_analytics.silver_vendors  (20 rows)
16:31:08  INFO  Loaded  <- healthcare_procurement.procurement_analytics.silver_purchase_orders  (481 rows)
16:31:09  INFO  Loaded  <- healthcare_procurement.procurement_analytics.silver_line_items  (1657 rows)
16:31:10  INFO  Loaded  <- healthcare_procurement.procurement_analytics.silver_spend_taxonomy  (39 rows)


---
### Taxonomy Join

Joins `silver_line_items` to `silver_spend_taxonomy` using a case-insensitive keyword substring match on `item_description`.


Algorithm:

- Convert the raw `item_description` into **lowercase** text.

- Join the taxonomy table by checking if the description contains the `keyword`.

- Measure the **character length** of the matched keyword.

- Group the matches by the line identification number, `line_id`.

- Sort the matches to keep only the **longest** keyword.

- Assign the label "Unclassified" to any item without a match.


**Leading Result**: The output produces a clean line items table. Every row now has a standardized `category` and `subcategory` attached to it.


In [0]:
from pyspark.sql.window import Window

# This function standardizes messy item descriptions. 
# It assigns official medical categories to raw purchase lines.

lines_with_tax = (
    lines_s
    .withColumn("item_desc_lower", F.lower(F.col("item_description")))
    .join(
        taxonomy_s.select("keyword", "subcategory", "category"),
        on=F.col("item_desc_lower").contains(F.col("keyword")),
        how="left",
    )
    .withColumn("_kw_len",
        F.when(F.col("keyword").isNotNull(), F.length("keyword")).otherwise(0))
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("line_id").orderBy(F.col("_kw_len").desc())))
    .filter(F.col("_rn") == 1)
    .drop("item_desc_lower", "_kw_len", "_rn")
    .withColumn("subcategory", F.coalesce(F.col("subcategory"), F.lit("Unclassified")))
    .withColumn("category",    F.coalesce(F.col("category"),    F.lit("Unclassified")))
)

log.info("lines_with_taxonomy count: %d rows", lines_with_tax.count())
lines_with_tax.show(5, truncate=False)

16:31:23  INFO  lines_with_taxonomy count: 1657 rows


+--------+-------+-----------------------+--------+----------+---------------+-----------------+-----------------+
|line_id |po_id  |item_description       |quantity|unit_price|keyword        |subcategory      |category         |
+--------+-------+-----------------------+--------+----------+---------------+-----------------+-----------------+
|LI-00001|PO-0001|cleaning cloths        |39      |429.4     |cleaning cloths|Cleaning Supplies|Facilities       |
|LI-00002|PO-0001|face masks surgical    |15      |347.77    |face masks     |PPE              |Medical Supplies |
|LI-00003|PO-0001|metformin 850mg        |24      |100.36    |metformin      |Medications      |Pharmacy         |
|LI-00004|PO-0002|xray developer solution|150     |41.1      |xray           |Imaging          |Medical Equipment|
|LI-00005|PO-0002|iv bags 500ml          |143     |433.24    |NULL           |Unclassified     |Unclassified     |
+--------+-------+-----------------------+--------+----------+---------------+--

### Build `gold_spend_by_vendor`

Total approved spend per vendor, ranked. Joins `silver_purchase_orders` to `silver_vendors`.

Algorithm:

- Filter the purchase orders to include only **approved** `status`.

- Connect the vendor profile data using the vendor identification number, `vender_id`.

- Group the combined dataset by vendor details, `vendor_id`, `vendor_name`, `vendor_category`, `payment_terms`.

- Count the total number of orders, `total_pos`.

- Sum the total spending amount,`total_amount`.

- Calculate the average value of each order, `avg_po_value`.

- Rank the vendors from highest total spend to lowest, `spend_rank`.

**Leading Result**: The output reveals the top vendors. The vendor with the rank of one represents the largest portion of the hospital procurement budget.

In [0]:

# This function identifies the most valuable medical suppliers. 
# It calculates exact financial totals for every approved vendor.

gold_spend_by_vendor = (
    po_s
    .filter(F.col("status") == "Approved")
    .join(
        vendors_s.select("vendor_id", "vendor_name", "vendor_category", "payment_terms"),
        on="vendor_id", how="inner")
    .groupBy("vendor_id", "vendor_name", "vendor_category", "payment_terms")
    .agg(
        F.count("po_id").alias("total_pos"),
        F.round(F.sum("total_amount"), 2).alias("total_spend"),
        F.round(F.avg("total_amount"), 2).alias("avg_po_value"),
    )
    .withColumn("spend_rank", F.row_number().over(
        Window.orderBy(F.col("total_spend").desc())))
    .orderBy("spend_rank")
)

write_table(gold_spend_by_vendor, "gold_spend_by_vendor")
log.info("gold_spend_by_vendor count: %d rows", gold_spend_by_vendor.count())
gold_spend_by_vendor.show(10, truncate=False)

/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
16:31:37  INFO  Written -> healthcare_procurement.procurement_analytics.gold_spend_by_vendor  (20 rows)
16:31:38  INFO  gold_spend_by_vendor count: 20 rows


+---------+--------------------------+-----------------+-------------+---------+-----------+------------+----------+
|vendor_id|vendor_name               |vendor_category  |payment_terms|total_pos|total_spend|avg_po_value|spend_rank|
+---------+--------------------------+-----------------+-------------+---------+-----------+------------+----------+
|V008     |Siemens Healthineers      |Medical Equipment|Net 60       |23       |610086.12  |26525.48    |1         |
|V010     |3m Health Care            |Medical Supplies |Net 30       |20       |518225.4   |25911.27    |2         |
|V002     |Cardinal Health           |Medical Supplies |Net 60       |18       |510105.85  |28339.21    |3         |
|V016     |Staples Business Advantage|Administrative   |Net 30       |18       |493444.38  |27413.58    |4         |
|V001     |Medline Industries        |Medical Supplies |Net 30       |19       |480641.68  |25296.93    |5         |
|V019     |Abbott Laboratories       |Pharmacy         |Net 30  

### Build `gold_spend_by_department`

Aggregate total spend per department from approved purchase orders

Algorithm:

- Filter out any unapproved purchase orders, `col("status") == "Approved"`.

- Group the data by the department identification number, `department_id`.

- Count the total number of approved orders for each group, `total_amount`.

- Calculate the total monetary spend for each group,`total_spend`.

- Calculate the average order value for each group,`avg_po_value`.

- Order the final list from the highest total spend to the lowest, `col("total_spend").desc()`.

**Leading Result**: The output highlights the most expensive hospital departments. The top row shows the department consuming the most financial resources.

In [0]:
# This script tracks internal hospital budgets. 
# It measures how much money each medical department spends.

gold_spend_by_department = (
    po_s
    .filter(F.col("status") == "Approved")
    .groupBy("department_id")
    .agg(
        F.count("po_id").alias("total_pos"),
        F.round(F.sum("total_amount"), 2).alias("total_spend"),
        F.round(F.avg("total_amount"), 2).alias("avg_po_value"),
    )
    .orderBy(F.col("total_spend").desc())
)

write_table(gold_spend_by_department, "gold_spend_by_department")
log.info("gold_spend_by_department count: %d rows", gold_spend_by_department.count())
gold_spend_by_department.show(truncate=False)

16:31:45  INFO  Written -> healthcare_procurement.procurement_analytics.gold_spend_by_department  (20 rows)
16:31:46  INFO  gold_spend_by_department count: 20 rows


+-------------------+---------+-----------+------------+
|department_id      |total_pos|total_spend|avg_po_value|
+-------------------+---------+-----------+------------+
|Icu                |38       |1030197.38 |27110.46    |
|Emergency          |33       |852764.75  |25841.36    |
|Radiology          |33       |784214.11  |23764.06    |
|Pharmacy           |31       |759916.28  |24513.43    |
|Pediatrics         |34       |753647.11  |22166.09    |
|Oncology           |33       |742854.63  |22510.75    |
|Surgery            |27       |727795.54  |26955.39    |
|Cardiology         |27       |705115.28  |26115.38    |
|Facilities         |32       |699322.44  |21853.83    |
|Administration     |23       |604991.51  |26303.98    |
|Cardiology Dept    |6        |172290.2   |28715.03    |
|Emergency Dept     |4        |114001.57  |28500.39    |
|Icu Dept           |4        |113476.24  |28369.06    |
|Pharmacy Dept      |4        |108308.94  |27077.24    |
|Facilities Dept    |4        |

---
### Build `gold_monthly_trend`

Total approved spend aggregated by month using `po_date`. Reveals whether procurement spend is growing, shrinking, or seasonal over time.

Algorithm:

Select only the approved purchase orders, `col("status") == "Approved"`.

Extract the specific year from the order date,`po_date`.

Extract the specific month from the order date, `po_date`.

Combine the year and month into a new formatted text string, `year_month`.

Group all transactions by these time periods.

Sum the total spending and count the orders for each period, `total_amount`.

Sort the final table chronologically,`year` & `month`.

**Leading Result**: The output displays a chronological timeline. Administrators can immediately spot which specific month had the highest procurement spending.

In [0]:
from pyspark.sql.types import StringType

# This code reveals purchasing patterns over time. 
# It helps administrators see financial growth or seasonal changes.

gold_monthly_trend = (
    po_s
    .filter(F.col("status") == "Approved")
    .withColumn("year",  F.year("po_date"))
    .withColumn("month", F.month("po_date"))
    .withColumn("year_month",
        F.concat_ws("-",
            F.col("year").cast(StringType()),
            F.lpad(F.col("month").cast(StringType()), 2, "0")))
    .groupBy("year", "month", "year_month")
    .agg(
        F.count("po_id").alias("total_pos"),
        F.round(F.sum("total_amount"), 2).alias("total_spend"),
    )
    .orderBy("year", "month")
)

write_table(gold_monthly_trend, "gold_monthly_trend")
log.info("gold_monthly_trend count: %d rows", gold_monthly_trend.count())
gold_monthly_trend.show(truncate=False)

16:31:52  INFO  Written -> healthcare_procurement.procurement_analytics.gold_monthly_trend  (24 rows)
16:31:53  INFO  gold_monthly_trend count: 24 rows


+----+-----+----------+---------+-----------+
|year|month|year_month|total_pos|total_spend|
+----+-----+----------+---------+-----------+
|2023|1    |2023-01   |20       |455797.31  |
|2023|2    |2023-02   |5        |152955.91  |
|2023|3    |2023-03   |14       |459816.55  |
|2023|4    |2023-04   |13       |276181.69  |
|2023|5    |2023-05   |15       |343413.84  |
|2023|6    |2023-06   |16       |246357.37  |
|2023|7    |2023-07   |19       |494149.82  |
|2023|8    |2023-08   |15       |381851.47  |
|2023|9    |2023-09   |14       |303166.15  |
|2023|10   |2023-10   |12       |316584.79  |
|2023|11   |2023-11   |10       |172359.44  |
|2023|12   |2023-12   |13       |295029.53  |
|2024|1    |2024-01   |14       |357961.99  |
|2024|2    |2024-02   |14       |374766.63  |
|2024|3    |2024-03   |17       |358917.84  |
|2024|4    |2024-04   |16       |393668.96  |
|2024|5    |2024-05   |17       |398223.1   |
|2024|6    |2024-06   |22       |530003.22  |
|2024|7    |2024-07   |14       |4

---
### Build`gold_payment_cycle`

Calculates `days_to_pay` (paid_date minus po_date) per approved PO, extracts the payment terms threshold (e.g. Net 30 → 30 days), and flags late payments where days_to_pay exceeds the threshold.


Algorithm:

Filter for approved orders that have a valid payment date, `("status") == "Approved"`.

Connect the vendor table to access the official payment terms, `payment_terms`.

Calculate the actual days taken to pay by subtracting the order date from the payment date, `days_to_pay` = `datediff(F.col("paid_date"), F.col("po_date"))`.

Extract the allowed number of days from the vendor payment terms text `payment_terms`, `terms_days` equals numeric threshold extracted from `payment_terms` like **`Net 30`** becomes **`30`**.

Compare the actual days to the allowed days, `is_late` equals 1 if `days_to_pay` is greater than `terms_days` else 0.

Create a flag with a value of one if the payment was late.

**Leading Result*: The output identifies specific late payments. The table highlights exactly which orders exceeded the agreed vendor timeline.

In [0]:
# This function analyzes payment efficiency. 
# It checks if the hospital pays its vendor bills on time.

gold_payment_cycle = (
    po_s
    .filter(F.col("status") == "Approved")
    .filter(F.col("paid_date").isNotNull())
    .join(
        vendors_s.select("vendor_id", "vendor_name", "payment_terms"),
        on="vendor_id", how="inner")
    .withColumn("days_to_pay", F.datediff(F.col("paid_date"), F.col("po_date")))
    .withColumn("terms_days",
        F.regexp_extract(F.col("payment_terms"), r"(\d+)", 1).cast(IntegerType()))
    .withColumn("is_late",
        F.when(F.col("days_to_pay") > F.col("terms_days"), 1).otherwise(0))
    .select("po_id", "vendor_id", "vendor_name", "payment_terms",
            "terms_days", "po_date", "paid_date", "days_to_pay", "is_late")
    .orderBy("vendor_name", "po_date")
)

write_table(gold_payment_cycle, "gold_payment_cycle")
log.info("gold_payment_cycle count: %d rows", gold_payment_cycle.count())
gold_payment_cycle.show(10, truncate=False)

16:31:59  INFO  Written -> healthcare_procurement.procurement_analytics.gold_payment_cycle  (321 rows)
16:32:00  INFO  gold_payment_cycle count: 321 rows


+-------+---------+--------------+-------------+----------+----------+----------+-----------+-------+
|po_id  |vendor_id|vendor_name   |payment_terms|terms_days|po_date   |paid_date |days_to_pay|is_late|
+-------+---------+--------------+-------------+----------+----------+----------+-----------+-------+
|PO-0403|V010     |3m Health Care|Net 30       |30        |2023-01-11|2023-03-02|50         |1      |
|PO-0328|V010     |3m Health Care|Net 30       |30        |2023-03-03|2023-04-11|39         |1      |
|PO-0085|V010     |3m Health Care|Net 30       |30        |2023-05-09|2023-07-20|72         |1      |
|PO-0427|V010     |3m Health Care|Net 30       |30        |2023-05-13|2023-06-03|21         |0      |
|PO-0275|V010     |3m Health Care|Net 30       |30        |2023-05-23|2023-06-21|29         |0      |
|PO-0271|V010     |3m Health Care|Net 30       |30        |2023-07-27|2023-09-02|37         |1      |
|PO-0419|V010     |3m Health Care|Net 30       |30        |2023-08-16|2023-10-04|4

---
### Build `gold_spend_by_taxonomy`

Aggregates spend by category and subcategory using the taxonomy-enriched line items from the taxonomy join above. Joins to `silver_purchase_orders` to filter for approved POs only.

Algorithm:

Take the categorized line items from the first step, `lines_with_tax`.

Connect these items to the approved purchase orders, `col("status") == "Approved"`.

Group the combined data by the broad `category` and specific `subcategory`.

Count the total number of individual items purchased, `quantity`.

Calculate the total spend by multiplying quantities by unit prices, `F.col("quantity") * F.col("unit_price")`.

Count the unique purchase orders for each group, `distinct_pos`.

Sort the results by the highest `total_spend`.

**Leading Result**: The output provides a clear financial breakdown of medical supplies. The top row reveals the specific item category that requires the largest budget.

In [0]:
# This code categorizes overall hospital spending. 
# It shows exactly what types of medical supplies cost the most money.

gold_spend_by_taxonomy = (
    lines_with_tax
    .join(
        po_s.filter(F.col("status") == "Approved")
            .select("po_id", "total_amount", "department_id"),
        on="po_id", how="inner")
    .groupBy("category", "subcategory")
    .agg(
        F.count("line_id").alias("total_line_items"),
        F.round(F.sum(F.col("quantity") * F.col("unit_price")), 2).alias("total_spend"),
        F.round(F.avg(F.col("quantity") * F.col("unit_price")), 2).alias("avg_line_value"),
        F.countDistinct("po_id").alias("distinct_pos"),
    )
    .orderBy(F.col("total_spend").desc())
)

write_table(gold_spend_by_taxonomy, "gold_spend_by_taxonomy")
log.info("gold_spend_by_taxonomy count: %d rows", gold_spend_by_taxonomy.count())
gold_spend_by_taxonomy.show(truncate=False)

16:32:09  INFO  Written -> healthcare_procurement.procurement_analytics.gold_spend_by_taxonomy  (10 rows)


### Gold — Verify Five Tables




In [0]:
print("--- GOLD LAYER SUMMARY ---")

# Create a list of the five new table names.
tables = [
    "gold_spend_by_vendor",
    "gold_spend_by_department",
    "gold_monthly_trend",
    "gold_payment_cycle",
    "gold_spend_by_taxonomy"
]

# Loop to verify each table one by one.
for tbl in tables:
    try:
        df = spark.table(tbl)
        cnt = df.count()
        print(f"\n  {tbl}")
        print(f"    Rows: {cnt}")
    except Exception as error:
        print(f"\n  {tbl}")
        print("    Status: Missing or Failed to Load. Need back to check if the table is saved successfully!")

print("\n" + "-" * 28)
print("\n  Gold complete. Run 04_eda_analysis.py next.")


---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
File <command-5032195412597675>, line 14
     12 # Loop to verify each table one by one.
     13 for tbl in tables:
---> 14     write_table(tbl, "tbl")
     15     df = spark.table(tbl)
     16     cnt = df.count()

File <command-5032195412597659>, line 4, in write_table(df, table_name, mode)
      2 """Write a Spark DataFrame as a Delta table in the active schema."""
      3 full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
----> 4 df.write.format("delta").mode(mode).saveAsTable(full_name)
      5 log.info("Written -> %s  (%d rows)", full_name, df.count())

AttributeError: 'str' object has no attribute 'write'